# Transformer-AR — canonical training

native autoregressive next-move head. Board size is selected once below and defaults to **12×12**.

The model is an 8-layer, 8-head causal Transformer with d_model=512 and dropout=0.1. The native bias-free LM head is trained jointly with the encoder.

The default smoke profile writes to a separate directory and cannot overwrite the full run.

In [ ]:
# Only these controls normally change.
BOARD_SIZE = 12                 # 8, 12, or 16
RUN_PROFILE = "smoke"           # "smoke" or "full"
SEED = 42
RESUME_CHECKPOINT = None        # synced latest.pt, or None
CONFIRM_FULL_RUN = ""           # set to "TRAIN" for a full run

assert BOARD_SIZE in {8, 12, 16}
assert RUN_PROFILE in {"smoke", "full"}
if RUN_PROFILE == "full":
    assert CONFIRM_FULL_RUN == "TRAIN", "Set CONFIRM_FULL_RUN = 'TRAIN'."

## Environment

Mount the Drive backup, install this final repository, and keep all large artifacts in `Master_Thesis_Artifacts`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os
import subprocess
import sys

PROJECT_ROOT = Path(
    "/content/drive/Othercomputers/MyLaptop/Master_Thesis_Code_Final"
)
assert (PROJECT_ROOT / "pyproject.toml").is_file(), PROJECT_ROOT
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", str(PROJECT_ROOT)],
    check=True,
)
os.chdir(PROJECT_ROOT)
print("PROJECT_ROOT:", PROJECT_ROOT)

## Resolve and validate the run

This reads the locked protocol and the primary path in the evaluation registry. No model is constructed here.

In [ ]:
from dataclasses import asdict
from othello_thesis.experiments import (
    build_ar_train_config,
    resolve_run_layout,
    validate_canonical_corpus,
)

layout = resolve_run_layout(
    "transformer", "ar", BOARD_SIZE, profile=RUN_PROFILE
)
corpus = validate_canonical_corpus(layout.data_dir, BOARD_SIZE)
cfg = build_ar_train_config(
    "transformer",
    BOARD_SIZE,
    layout,
    profile=RUN_PROFILE,
    seed=SEED,
    resume=RESUME_CHECKPOINT,
)
print("data:", layout.data_dir)
print("local output:", layout.local_out_dir)
print("Drive output:", layout.drive_out_dir)
print("resolved config:")
asdict(cfg)

## Visible model definition

The factory below is the factory used by training. It is not called in this cell. This preserves the exact global RNG trajectory. 

In [ ]:
import inspect
import torch
from othello_thesis.models.transformer import GPTConfig, OthelloGPT

def build_transformer_ar(model_config: GPTConfig) -> torch.nn.Module:
    """The trainer calls this factory once, after seeding."""
    assert model_config.n_layers == 8
    assert model_config.n_heads == 8
    assert model_config.d_model == 512
    return OthelloGPT(model_config)

# Display the real constructor without instantiating a preview model.
print(inspect.getsource(OthelloGPT.__init__))

## Train

This is the only cell that starts training and constructs a model. `smoke` and `full` use separate output paths. For resume, set `RESUME_CHECKPOINT` before resolving the config.

In [ ]:
from othello_thesis.training.transformer_ar import run_training

print(
    f"START {RUN_PROFILE}: TRANSFORMER-AR "
    f"{BOARD_SIZE}x{BOARD_SIZE}",
    flush=True,
)
run_training(cfg, model_factory=build_transformer_ar)
print("TRAINING FINISHED", flush=True)